# Tasks 4–6 — Preparation & Feature Engineering (Papermill/Prefect-safe)
Reads validated data, engineers a minimal common feature set, and saves
`data/transformed/churn_features.csv` with a `churn` label and `event_timestamp`.

In [1]:
import os, pandas as pd, numpy as np
from datetime import datetime
os.makedirs('data/transformed', exist_ok=True)

telco_path = 'data/validated/telco.csv'
gen_path = 'data/validated/generic.csv'
telco = pd.read_csv(telco_path)
gen = pd.read_csv(gen_path)

# Harmonize labels
telco['churn'] = (telco.get('Churn','No').astype(str).str.lower() == 'yes').astype(int)
gen['churn'] = gen.get('Exited', 0).astype(int)

def telco_to_common(df):
    out = pd.DataFrame()
    out['customer_id'] = df.get('customerID', pd.Series(range(1, len(df)+1))).astype(str)
    out['customer_tenure_days'] = (df.get('tenure', 1)*30).astype(int)
    out['total_spend_last_6m'] = (df.get('MonthlyCharges', 0)*6).astype(float)
    out['login_frequency_30d'] = np.random.poisson(lam=8, size=len(df))
    out['support_tickets_90d'] = np.random.poisson(lam=1.2, size=len(df))
    out['is_premium'] = (df.get('Contract','Month-to-month').astype(str) != 'Month-to-month').astype(int) if 'Contract' in df.columns else 0
    out['churn'] = df['churn'].astype(int)
    out['event_timestamp'] = pd.Timestamp('2025-06-30')
    return out

def gen_to_common(df):
    out = pd.DataFrame()
    out['customer_id'] = df.get('CustomerId', pd.Series(range(1, len(df)+1))).astype(str)
    out['customer_tenure_days'] = (df.get('Age', 30)*12).astype(int)
    out['total_spend_last_6m'] = (df.get('Balance', 0)/10).astype(float)
    out['login_frequency_30d'] = np.random.poisson(lam=6, size=len(df))
    out['support_tickets_90d'] = np.random.poisson(lam=0.8, size=len(df))
    out['is_premium'] = (df.get('CreditScore', 650) > 700).astype(int)
    out['churn'] = df['churn'].astype(int)
    out['event_timestamp'] = pd.Timestamp('2025-06-30')
    return out

telco_common = telco_to_common(telco)
gen_common = gen_to_common(gen)
features = pd.concat([telco_common, gen_common], ignore_index=True)

# Type casting
features['customer_id'] = features['customer_id'].astype(str)
for col in ['customer_tenure_days','login_frequency_30d','support_tickets_90d','is_premium','churn']:
    features[col] = features[col].astype(int)

out_path = 'data/transformed/churn_features.csv'
features.to_csv(out_path, index=False)
print('✅ Wrote', out_path)
print('\n=== Preview ===')
print(features.head().to_string(index=False))


✅ Wrote data/transformed/churn_features.csv

=== Preview ===
customer_id  customer_tenure_days  total_spend_last_6m  login_frequency_30d  support_tickets_90d  is_premium  churn event_timestamp
          1                    30                179.1                   10                    0           0      0      2025-06-30
          2                  1020                514.2                    7                    2           0      0      2025-06-30
          3                   360                341.7                    7                    0           0      1      2025-06-30
          1                   480               6000.0                    5                    1           0      0      2025-06-30
          2                   360              12000.0                    7                    0           1      1      2025-06-30
